# 02 — Model Experiments

Mirrors `src/models/train.py` in an interactive notebook for model comparison and selection.

**Models compared:** Logistic Regression vs Random Forest  
**Selection criterion:** ROC-AUC (tie-break: F1)

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path when running from notebooks/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.base import clone

from src import config
from src.data.ingestion import load_raw
from src.data.preprocessing import clean, split
from src.features.pipeline import build_preprocessor
from src.models.train import build_models, train_model, cross_validate_model, select_best
from src.models.evaluate import evaluate, plot_confusion_matrix, plot_roc_curve, plot_pr_curve

print('Imports OK')

## 1. Load & Prepare Data

In [ ]:
df = load_raw(config.RAW_DATA_PATH)
df_clean = clean(df)

X_train, X_test, y_train, y_test = split(
    df_clean,
    target_col=config.TARGET_COL,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')

## 2. Train Both Models

In [ ]:
models_dict = build_models(config.RANDOM_STATE)
results = {}

for model_name, model_estimator in models_dict.items():
    preprocessor = clone(build_preprocessor(config.NUMERIC_COLS, config.CATEGORICAL_COLS))
    pipeline = train_model(model_estimator, preprocessor, X_train, y_train)
    
    cv_metrics = cross_validate_model(pipeline, X_train, y_train, cv=5)
    metrics = evaluate(pipeline, X_test, y_test)
    
    results[model_name] = {
        'pipeline': pipeline,
        'metrics': metrics,
        'cv_metrics': cv_metrics,
    }
    print(f'{model_name}: accuracy={metrics["accuracy"]:.4f}, roc_auc={metrics["roc_auc"]:.4f}')

## 3. Comparison Table

In [ ]:
comparison = []
for name, res in results.items():
    row = {'model': name}
    row.update(res['metrics'])
    comparison.append(row)

df_comparison = pd.DataFrame(comparison).set_index('model')
df_comparison = df_comparison.round(4)
print('=== Holdout Test Metrics ===')
display(df_comparison)

In [ ]:
cv_comparison = []
for name, res in results.items():
    row = {'model': name}
    row.update({f'cv_{k}': v for k, v in res['cv_metrics'].items()})
    cv_comparison.append(row)

df_cv = pd.DataFrame(cv_comparison).set_index('model')
df_cv = df_cv.round(4)
print('=== 5-Fold CV Metrics (Training Data) ===')
display(df_cv)

## 4. Select Best Model

In [ ]:
best_name, best_pipeline, best_metrics = select_best(results)
print(f'Best model: {best_name}')
print(f'  ROC-AUC : {best_metrics["roc_auc"]:.4f}')
print(f'  F1      : {best_metrics["f1"]:.4f}')
print(f'  Accuracy: {best_metrics["accuracy"]:.4f}')

## 5. Diagnostic Plots for Best Model

In [ ]:
import matplotlib
matplotlib.use('Agg')

screenshots_dir = PROJECT_ROOT / 'screenshots'
screenshots_dir.mkdir(exist_ok=True)

y_pred = best_pipeline.predict(X_test)
y_score = best_pipeline.predict_proba(X_test)[:, 1]

cm_path = plot_confusion_matrix(y_test, y_pred, screenshots_dir / f'{best_name}_confusion_matrix.png')
roc_path = plot_roc_curve(y_test, y_score, screenshots_dir / f'{best_name}_roc_curve.png')
pr_path = plot_pr_curve(y_test, y_score, screenshots_dir / f'{best_name}_pr_curve.png')

print(f'Plots saved: {cm_path.name}, {roc_path.name}, {pr_path.name}')

## 6. Feature Importances (Random Forest)

In [ ]:
from src.features.pipeline import get_feature_names
import numpy as np

rf_pipeline = results.get('random_forest', {}).get('pipeline')
if rf_pipeline is not None:
    feature_names = get_feature_names(rf_pipeline.named_steps['preprocessor'])
    importances = rf_pipeline.named_steps['classifier'].feature_importances_
    indices = np.argsort(importances)[::-1][:10]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(
        [feature_names[i] for i in indices[::-1]],
        importances[indices[::-1]],
    )
    ax.set_title('Random Forest — Top 10 Feature Importances')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(screenshots_dir / 'rf_feature_importances.png', dpi=100)
    plt.show()
    print('Feature importance plot saved.')
else:
    print('Random Forest not trained.')

## Summary

| Model | Selection criterion | Result |
|---|---|---|
| Logistic Regression | ROC-AUC | see table above |
| Random Forest | ROC-AUC | see table above |

**Winner:** determined by `select_best()` → highest ROC-AUC, F1 as tie-breaker.